# DocArray

>[DocArray](https://github.com/docarray/docarray) 是一款多功能、开源的工具，用于管理您的多模态数据。它允许您根据需要塑造数据，并提供使用各种文档索引后端存储和搜索数据的灵活性。此外，它还能带来更多优势——您可以利用您的 `DocArray` 文档索引来创建一个 `DocArrayRetriever`，并构建出色的 Langchain 应用！

本 Notebook 分为两个部分。 [第一部分](#document-index-backends) 介绍了所有五个支持的文档索引后端。它提供了有关设置和索引每个后端的指南，并指导您如何构建 `DocArrayRetriever` 来查找相关文档。
在 [第二部分](#movie-retrieval-using-hnswdocumentindex) 中，我们将选择其中一个后端，并通过一个基本示例来说明如何使用它。

## 文档索引后端

This document describes the backends that can be used for document indexing.

本文档介绍了可用于文档索引的后端。

### Elasticsearch

Elasticsearch is a distributed, RESTful search and analytics engine capable of solving a growing number of use cases. As the heart of the Elastic Stack, it may be used to power your search, logs, and systems integrity use cases.

Elasticsearch 是一个分布式、RESTful 的搜索和分析引擎，能够解决日益增长的用例。作为 Elastic Stack 的核心，它可以为您的搜索、日志和系统完整性用例提供支持。

Elasticsearch is a popular choice for document indexing due to its powerful search capabilities, scalability, and flexibility.

由于其强大的搜索功能、可扩展性和灵活性，Elasticsearch 是文档索引的流行选择。

#### Configuration

#### 配置

To use Elasticsearch as a backend, you need to provide the connection details.

要使用 Elasticsearch 作为后端，您需要提供连接详细信息。

```json
{
  "backend": "elasticsearch",
  "host": "localhost",
  "port": 9200,
  "index": "documents"
}
```

-   `backend`: Specifies the backend to use, which is "elasticsearch" in this case.
    -   `backend`: 指定要使用的后端，在本例中为“elasticsearch”。
-   `host`: The hostname or IP address of your Elasticsearch instance.
    -   `host`: Elasticsearch 实例的主机名或 IP 地址。
-   `port`: The port number your Elasticsearch instance is listening on.
    -   `port`: Elasticsearch 实例正在监听的端口号。
-   `index`: The name of the Elasticsearch index to use for storing documents.
    -   `index`: 用于存储文档的 Elasticsearch 索引的名称。

### Solr

Apache Solr is an open-source enterprise search platform. Its major features include comprehensive full-text search, hit highlighting, **faceted search**, **rich document (Word, PDF, etc.) handling**, and geospatial search. It is highly scalable and distributed, with a web-based administration interface.

Apache Solr 是一个开源的企业搜索平台。其主要功能包括全面的全文搜索、命中高亮、**分面搜索**、**富文档（Word、PDF 等）处理**以及地理空间搜索。它具有高度的可扩展性和分布式特性，并提供基于 Web 的管理界面。

Solr is another excellent choice for document indexing, offering similar capabilities to Elasticsearch.

Solr 是文档索引的另一个绝佳选择，提供与 Elasticsearch 类似的功能。

#### Configuration

#### 配置

To use Solr as a backend, you need to provide the connection details.

要使用 Solr 作为后端，您需要提供连接详细信息。

```json
{
  "backend": "solr",
  "host": "localhost",
  "port": 8983,
  "core": "documents"
}
```

-   `backend`: Specifies the backend to use, which is "solr" in this case.
    -   `backend`: 指定要使用的后端，在本例中为“solr”。
-   `host`: The hostname or IP address of your Solr instance.
    -   `host`: Solr 实例的主机名或 IP 地址。
-   `port`: The port number your Solr instance is listening on.
    -   `port`: Solr 实例正在监听的端口号。
-   `core`: The name of the Solr core to use for storing documents.
    -   `core`: 用于存储文档的 Solr 核心的名称。

### Other Backends

Other backends might be supported in the future, such as:

### 其他后端

未来可能会支持其他后端，例如：

-   **Meilisearch**: A fast, open-source, and easy-to-use full-text search engine.
    -   **Meilisearch**: 一个快速、开源且易于使用的全文搜索引擎。
-   **Typesense**: A fast, typo-tolerant, and relevant search engine.
    -   **Typesense**: 一个快速、容错（可处理拼写错误）且相关的搜索引擎。

The configuration for these backends would follow a similar pattern, specifying the `backend` type and relevant connection details.

这些后端的配置将遵循类似的模式，指定 `backend` 类型和相关的连接详细信息。

In [2]:
import random

from docarray import BaseDoc
from docarray.typing import NdArray
from langchain_community.embeddings import FakeEmbeddings
from langchain_community.retrievers import DocArrayRetriever

embeddings = FakeEmbeddings(size=32)

在开始构建索引之前，定义文档模式非常重要。这将决定您的文档将包含哪些字段以及每个字段将存储什么类型的数据。

在本演示中，我们将创建一个包含“title”（字符串）、“title_embedding”（numpy 数组）、“year”（整数）和“color”（字符串）的随机模式。

In [2]:
class MyDoc(BaseDoc):
    title: str
    title_embedding: NdArray[32]
    year: int
    color: str

### InMemoryExactNNIndex

`InMemoryExactNNIndex` 将所有文档存储在内存中。对于小型数据集来说，这是一个很好的起点，您可能不想启动数据库服务器。

在此了解更多：https://docs.docarray.org/user_guide/storing/index_in_memory/

In [3]:
from docarray.index import InMemoryExactNNIndex

# initialize the index
db = InMemoryExactNNIndex[MyDoc]()
# index data
db.index(
    [
        MyDoc(
            title=f"My document {i}",
            title_embedding=embeddings.embed_query(f"query {i}"),
            year=i,
            color=random.choice(["red", "green", "blue"]),
        )
        for i in range(100)
    ]
)
# optionally, you can create a filter query
filter_query = {"year": {"$lte": 90}}

In [4]:
# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="title_embedding",
    content_field="title",
    filters=filter_query,
)

# find the relevant document
doc = retriever.invoke("some query")
print(doc)

[Document(page_content='My document 56', metadata={'id': '1f33e58b6468ab722f3786b96b20afe6', 'year': 56, 'color': 'red'})]


### HnswDocumentIndex

`HnswDocumentIndex` 是一个轻量级的 Document Index 实现，完全在本地运行，最适合处理小型到中型数据集。它将向量存储在磁盘上的 [hnswlib](https://github.com/nmslib/hnswlib) 中，并将所有其他数据存储在 [SQLite](https://www.sqlite.org/index.html) 中。

在此处了解更多信息：https://docs.docarray.org/user_guide/storing/index_hnswlib/

In [5]:
from docarray.index import HnswDocumentIndex

# initialize the index
db = HnswDocumentIndex[MyDoc](work_dir="hnsw_index")

# index data
db.index(
    [
        MyDoc(
            title=f"My document {i}",
            title_embedding=embeddings.embed_query(f"query {i}"),
            year=i,
            color=random.choice(["red", "green", "blue"]),
        )
        for i in range(100)
    ]
)
# optionally, you can create a filter query
filter_query = {"year": {"$lte": 90}}

In [6]:
# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="title_embedding",
    content_field="title",
    filters=filter_query,
)

# find the relevant document
doc = retriever.invoke("some query")
print(doc)

[Document(page_content='My document 28', metadata={'id': 'ca9f3f4268eec7c97a7d6e77f541cb82', 'year': 28, 'color': 'red'})]


### WeaviateDocumentIndex

`WeaviateDocumentIndex` 是一个基于 [Weaviate](https://weaviate.io/) 向量数据库构建的文档索引。

在此了解更多：https://docs.docarray.org/user_guide/storing/index_weaviate/

In [7]:
# There's a small difference with the Weaviate backend compared to the others.
# Here, you need to 'mark' the field used for vector search with 'is_embedding=True'.
# So, let's create a new schema for Weaviate that takes care of this requirement.

from pydantic import Field


class WeaviateDoc(BaseDoc):
    title: str
    title_embedding: NdArray[32] = Field(is_embedding=True)
    year: int
    color: str

In [8]:
from docarray.index import WeaviateDocumentIndex

# initialize the index
dbconfig = WeaviateDocumentIndex.DBConfig(host="http://localhost:8080")
db = WeaviateDocumentIndex[WeaviateDoc](db_config=dbconfig)

# index data
db.index(
    [
        MyDoc(
            title=f"My document {i}",
            title_embedding=embeddings.embed_query(f"query {i}"),
            year=i,
            color=random.choice(["red", "green", "blue"]),
        )
        for i in range(100)
    ]
)
# optionally, you can create a filter query
filter_query = {"path": ["year"], "operator": "LessThanEqual", "valueInt": "90"}

In [9]:
# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="title_embedding",
    content_field="title",
    filters=filter_query,
)

# find the relevant document
doc = retriever.invoke("some query")
print(doc)

[Document(page_content='My document 17', metadata={'id': '3a5b76e85f0d0a01785dc8f9d965ce40', 'year': 17, 'color': 'red'})]


### ElasticDocIndex

`ElasticDocIndex` 是一个基于 [ElasticSearch](https://github.com/elastic/elasticsearch) 构建的文档索引。

在此了解更多信息：[https://docs.docarray.org/user_guide/storing/index_elastic/](https://docs.docarray.org/user_guide/storing/index_elastic/)

In [10]:
from docarray.index import ElasticDocIndex

# initialize the index
db = ElasticDocIndex[MyDoc](
    hosts="http://localhost:9200", index_name="docarray_retriever"
)

# index data
db.index(
    [
        MyDoc(
            title=f"My document {i}",
            title_embedding=embeddings.embed_query(f"query {i}"),
            year=i,
            color=random.choice(["red", "green", "blue"]),
        )
        for i in range(100)
    ]
)
# optionally, you can create a filter query
filter_query = {"range": {"year": {"lte": 90}}}

In [11]:
# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="title_embedding",
    content_field="title",
    filters=filter_query,
)

# find the relevant document
doc = retriever.invoke("some query")
print(doc)

[Document(page_content='My document 46', metadata={'id': 'edbc721bac1c2ad323414ad1301528a4', 'year': 46, 'color': 'green'})]


### QdrantDocumentIndex

`QdrantDocumentIndex` 是一个基于 [Qdrant](https://qdrant.tech/) 向量数据库构建的文档索引。

在此处了解更多信息：[https://docs.docarray.org/user_guide/storing/index_qdrant/](https://docs.docarray.org/user_guide/storing/index_qdrant/)

In [12]:
from docarray.index import QdrantDocumentIndex
from qdrant_client.http import models as rest

# initialize the index
qdrant_config = QdrantDocumentIndex.DBConfig(path=":memory:")
db = QdrantDocumentIndex[MyDoc](qdrant_config)

# index data
db.index(
    [
        MyDoc(
            title=f"My document {i}",
            title_embedding=embeddings.embed_query(f"query {i}"),
            year=i,
            color=random.choice(["red", "green", "blue"]),
        )
        for i in range(100)
    ]
)
# optionally, you can create a filter query
filter_query = rest.Filter(
    must=[
        rest.FieldCondition(
            key="year",
            range=rest.Range(
                gte=10,
                lt=90,
            ),
        )
    ]
)

In [13]:
# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="title_embedding",
    content_field="title",
    filters=filter_query,
)

# find the relevant document
doc = retriever.invoke("some query")
print(doc)

[Document(page_content='My document 80', metadata={'id': '97465f98d0810f1f330e4ecc29b13d20', 'year': 80, 'color': 'blue'})]


## 使用 HnswDocumentIndex 进行电影检索

This document explains how to use `HnswDocumentIndex` to retrieve similar documents.

本文档将介绍如何使用 `HnswDocumentIndex` 来检索相似文档。

### Setup

### 设置

```python
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.hnswlib import HNSWLibVectorStore
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Set embedding model
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Load documents
documents = SimpleDirectoryReader("./data").load_data()

# Create HNSWLibVectorStore
hnsw_vector_store = HNSWLibVectorStore(space="cosine", dim=768) # dim must match embed_model dimension

# Create StorageContext
storage_context = StorageContext.from_defaults(vector_store=hnsw_vector_store)

# Create VectorStoreIndex
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
```

```python
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.hnswlib import HNSWLibVectorStore
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 设置嵌入模型
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 加载文档
documents = SimpleDirectoryReader("./data").load_data()

# 创建 HNSWLibVectorStore
hnsw_vector_store = HNSWLibVectorStore(space="cosine", dim=768) # dim 必须与 embed_model 的维度匹配

# 创建 StorageContext
storage_context = StorageContext.from_defaults(vector_store=hnsw_vector_store)

# 创建 VectorStoreIndex
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
```

### Querying

### 查询

```python
# Create a query engine
query_engine = index.as_query_engine()

# Query the index
response = query_engine.query("What are the movies about?")

# Print the response
print(response)
```

```python
# 创建查询引擎
query_engine = index.as_query_engine()

# 查询索引
response = query_engine.query("电影是关于什么的？")

# 打印响应
print(response)
```

### Persistence

### 持久化

You can persist the index to disk.

您可以将索引持久化到磁盘。

```python
# Persist the index
index.storage_context.persist("./storage")
```

```python
# 持久化索引
index.storage_context.persist("./storage")
```

To load the index later:

稍后加载索引：

```python
# Load the index
storage_context = StorageContext.from_defaults(persist_dir="./storage")
load_index = VectorStoreIndex.from_documents(
    [], storage_context=storage_context
)
```

```python
# 加载索引
storage_context = StorageContext.from_defaults(persist_dir="./storage")
load_index = VectorStoreIndex.from_documents(
    [], storage_context=storage_context
)
```

### Customizing HNSW Parameters

### 自定义 HNSW 参数

You can customize HNSW parameters such as `ef_construction` and `M`.

您可以自定义 HNSW 参数，例如 `ef_construction` 和 `M`。

```python
hnsw_vector_store = HNSWLibVectorStore(
    space="cosine",
    dim=768,
    # HNSW parameters
    ef_construction=100,
    M=16,
)
```

```python
hnsw_vector_store = HNSWLibVectorStore(
    space="cosine",
    dim=768,
    # HNSW 参数
    ef_construction=100,
    M=16,
)

In [14]:
movies = [
    {
        "title": "Inception",
        "description": "A thief who steals corporate secrets through the use of dream-sharing technology is given the task of planting an idea into the mind of a CEO.",
        "director": "Christopher Nolan",
        "rating": 8.8,
    },
    {
        "title": "The Dark Knight",
        "description": "When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.",
        "director": "Christopher Nolan",
        "rating": 9.0,
    },
    {
        "title": "Interstellar",
        "description": "Interstellar explores the boundaries of human exploration as a group of astronauts venture through a wormhole in space. In their quest to ensure the survival of humanity, they confront the vastness of space-time and grapple with love and sacrifice.",
        "director": "Christopher Nolan",
        "rating": 8.6,
    },
    {
        "title": "Pulp Fiction",
        "description": "The lives of two mob hitmen, a boxer, a gangster's wife, and a pair of diner bandits intertwine in four tales of violence and redemption.",
        "director": "Quentin Tarantino",
        "rating": 8.9,
    },
    {
        "title": "Reservoir Dogs",
        "description": "When a simple jewelry heist goes horribly wrong, the surviving criminals begin to suspect that one of them is a police informant.",
        "director": "Quentin Tarantino",
        "rating": 8.3,
    },
    {
        "title": "The Godfather",
        "description": "An aging patriarch of an organized crime dynasty transfers control of his empire to his reluctant son.",
        "director": "Francis Ford Coppola",
        "rating": 9.2,
    },
]

In [15]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

OpenAI API Key: ········


In [16]:
from docarray import BaseDoc, DocList
from docarray.typing import NdArray
from langchain_openai import OpenAIEmbeddings


# define schema for your movie documents
class MyDoc(BaseDoc):
    title: str
    description: str
    description_embedding: NdArray[1536]
    rating: float
    director: str


embeddings = OpenAIEmbeddings()


# get "description" embeddings, and create documents
docs = DocList[MyDoc](
    [
        MyDoc(
            description_embedding=embeddings.embed_query(movie["description"]), **movie
        )
        for movie in movies
    ]
)

In [17]:
from docarray.index import HnswDocumentIndex

# initialize the index
db = HnswDocumentIndex[MyDoc](work_dir="movie_search")

# add data
db.index(docs)

### 常规检索器

In [18]:
from langchain_community.retrievers import DocArrayRetriever

# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="description_embedding",
    content_field="description",
)

# find the relevant document
doc = retriever.invoke("movie about dreams")
print(doc)

[Document(page_content='A thief who steals corporate secrets through the use of dream-sharing technology is given the task of planting an idea into the mind of a CEO.', metadata={'id': 'f1649d5b6776db04fec9a116bbb6bbe5', 'title': 'Inception', 'rating': 8.8, 'director': 'Christopher Nolan'})]


### 带过滤器的检索器

In [19]:
from langchain_community.retrievers import DocArrayRetriever

# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="description_embedding",
    content_field="description",
    filters={"director": {"$eq": "Christopher Nolan"}},
    top_k=2,
)

# find relevant documents
docs = retriever.invoke("space travel")
print(docs)

[Document(page_content='Interstellar explores the boundaries of human exploration as a group of astronauts venture through a wormhole in space. In their quest to ensure the survival of humanity, they confront the vastness of space-time and grapple with love and sacrifice.', metadata={'id': 'ab704cc7ae8573dc617f9a5e25df022a', 'title': 'Interstellar', 'rating': 8.6, 'director': 'Christopher Nolan'}), Document(page_content='A thief who steals corporate secrets through the use of dream-sharing technology is given the task of planting an idea into the mind of a CEO.', metadata={'id': 'f1649d5b6776db04fec9a116bbb6bbe5', 'title': 'Inception', 'rating': 8.8, 'director': 'Christopher Nolan'})]


### 检索器与 MMR 搜索

In [20]:
from langchain_community.retrievers import DocArrayRetriever

# create a retriever
retriever = DocArrayRetriever(
    index=db,
    embeddings=embeddings,
    search_field="description_embedding",
    content_field="description",
    filters={"rating": {"$gte": 8.7}},
    search_type="mmr",
    top_k=3,
)

# find relevant documents
docs = retriever.invoke("action movies")
print(docs)

[Document(page_content="The lives of two mob hitmen, a boxer, a gangster's wife, and a pair of diner bandits intertwine in four tales of violence and redemption.", metadata={'id': 'e6aa313bbde514e23fbc80ab34511afd', 'title': 'Pulp Fiction', 'rating': 8.9, 'director': 'Quentin Tarantino'}), Document(page_content='A thief who steals corporate secrets through the use of dream-sharing technology is given the task of planting an idea into the mind of a CEO.', metadata={'id': 'f1649d5b6776db04fec9a116bbb6bbe5', 'title': 'Inception', 'rating': 8.8, 'director': 'Christopher Nolan'}), Document(page_content='When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.', metadata={'id': '91dec17d4272041b669fd113333a65f7', 'title': 'The Dark Knight', 'rating': 9.0, 'director': 'Christopher Nolan'})]
